In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 10 · 数值梯度检验（第 1 阶段）

对应任务书：§1.1（检验器）、§1.2（三个通用检查）、§1.3（纯 SGD 骨架）、§2.x 各模块的 float64 检验。

本 notebook 的产出：
- [x] cell 1（已给，五个 notebook 一字不差，不要改）：bootstrap + 环境留痕 + 线程限制
- [x] **§1.1 工具自检**（cell 2）：`f(x)=sum(x**3)` 已知答案 PASS（rel_err ≈ 1e-10），
      故意写错一版判 FAIL —— 尺子本身先被验证过（`tests/grad_check.py`）
- [x] **h 扫描**（cell 3）：U 形曲线 → `report/figs/10_h_sweep.png`。
      截断误差段按 h² 下降、舍入误差段随 h 变小上升，最低点 h≈1e-5~1e-6
- [ ] **模块对拍汇总**（cell 4，脚本 `tests/test_modules.py`）—— 进度 **3/6**
      - [x] **Linear**：4 组 case（含 3 维/4 维输入、bias=False）全 PASS，
            前向 ≤2.2e-16、反向 ≤3.4e-16、数值梯度 ≤7.0e-11
      - [x] **GELU**：3 组 case 全 PASS（含 uniform(-4,4)、linspace(-6,6)、N(0,3)），
            前向 ≤8.9e-16、反向 ≤3.7e-15、数值梯度 ≤1.6e-10；
            与 erf 版最大偏差 **4.732e-04 @ x=−2.70**（任务书预期 ≈4.7e-4 @ −2.7 ✓）
      - [x] **CrossEntropyLoss**：3 组 case（含 N=1、1e3 大 logits）全 PASS，
            前向 Δ=0、dz ≤3.5e-18、数值梯度 ≤3.2e-9；
            手算 2×3 完全吻合（loss 0.753109，梯度差 5.6e-17）；
            **不减 max 的版本在同一组 logits 上得到 inf** —— 坑 1 的现场证据
- [x] **§1.3 纯 SGD 训练链路**（cell 5）：真实 CIFAR-100 的 4000 张子集，10 epoch，
      loss 4.172 → 2.813，train acc 8.9% → 33.2%，val acc 13.7%，0.29 s/epoch。
      **接线是通的** —— 之后 Adam 出问题时不用怀疑这里

约束（根 agent.md）：cell 只依赖 cell 1（+2）；公共逻辑进 `npnn/` 或 `tests/`，不要写在 notebook 里。

## cell 1 已给 / cell 2 你要写什么

**cell 1（上面那格，已写好，别改）—— 环境留痕 + bootstrap**，依次做四件事：
1. 向上搜索含 `npnn/` 的目录 → `sys.path`（不用 `cwd().parent`，从任何目录启动 Jupyter 都能跑）
2. 把项目根从 `sys.path` 摘掉 —— 物理上挡住「误 `import src`」
3. `config.limit_threads()` —— **必须在 import numpy 之前**（BLAS 首次 import 就锁死线程数）
4. 打印 `sys.executable` / `sys.prefix` / numpy 版本 / `TASKB_ROOT`（报告 B.1 的环境留痕）

**cell 2（你来写）—— 本 notebook 的导入**：`from tests.grad_check import ...`（记得把 `tests/` 也加进 path）、
`from npnn import ...`，以及本 notebook 要用的常量。

---
⚠️ 检验一律 float64；输入要随机且多组；非标量输出用随机线性组合标量化 —— 三条都违反的话，
检验会「通过」但什么也没验证。判据：<1e-6 通过 / 1e-6~1e-4 可疑 / >1e-4 有 bug。

In [ ]:
# --- cell 2 · §1.1 工具自检：先证明「尺子」本身是准的 ---
# 已知答案 f(x)=sum(x**3) → 解析梯度 3x**2；再故意写错一版，确认检验器会判 FAIL
import numpy as np
from tests.grad_check import check, format_result

rng = np.random.default_rng(0)
x = rng.standard_normal((3, 4))
f_cube = lambda ps: float(np.sum(ps[0] ** 3))

print('known answer: f(x)=sum(x**3), analytic 3*x**2')
print(format_result(check(f_cube, [x], [3.0 * x ** 2])))

print('\nnegative control: analytic 2*x**2 (deliberately wrong)')
print(format_result(check(f_cube, [x], [2.0 * x ** 2])))

In [ ]:
# --- cell 3 · §1.1 h 扫描：那条 U 形曲线（报告图）---
# 左边（h 大）是截断误差 O(h^2)，右边（h 小）是舍入误差 O(eps/h)，最低点在 h≈1e-5
import matplotlib.pyplot as plt
from config import TASKB_ROOT
from tests.grad_check import h_sweep

hs = np.logspace(-2, -8, 13)
errs = [err for _h, err in h_sweep(f_cube, [x], [3.0 * x ** 2], hs=tuple(hs))]

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.loglog(hs, errs, 'o-', lw=1.6, ms=4)
ax.axhline(1e-6, color='tab:green', ls='--', lw=1, label='pass threshold 1e-6')
ax.axvline(1e-5, color='tab:gray', ls=':', lw=1, label='h = 1e-5')
ax.set_xlabel('step size h'); ax.set_ylabel('max relative error')
ax.set_title('central-difference error vs step size')
ax.invert_xaxis()                      # 惯用画法：h 由大到小
ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(TASKB_ROOT / 'report' / 'figs' / '10_h_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

best_h, best_err = min(zip(hs, errs), key=lambda z: z[1])
print(f'最优 h ≈ {best_h:.0e}   最小 max_rel_err = {best_err:.2e}')
print('→ 这就是"h 不能太大也不能太小"的实测形状；判据阈值 1e-6 的来历在这里')

In [ ]:
# --- cell 4 · 单模块对拍结果（脚本 tests/test_modules.py）---
# 报告里的「模块验证汇总表」就是从这个输出里抄的
import subprocess, sys

r = subprocess.run([sys.executable, str(TASKB_ROOT / 'tests' / 'test_modules.py')],
                   capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout)
print('exit code:', r.returncode, '（0 = 全部 PASS）')

In [ ]:
# --- cell 5 · §1.3 极简训练链路（脚本 tests/test_training_loop.py，纯 SGD、不用 torch）---
# 目的：在 Adam 写出来之前，先证明「数据 → 前向 → loss → 反向 → 更新」这条链是通的。
# 它跑的是真实 CIFAR-100 的 4000 张子集 + 我们自己的 Linear/GELU/CE。
import subprocess, sys

r = subprocess.run([sys.executable, str(TASKB_ROOT / 'tests' / 'test_training_loop.py')],
                   capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout)
print('exit code:', r.returncode, '（0 = 通过）')